# ARC Prize 2026 - ARC-AGI-2

Offline, task-level symbolic program synthesis. The source package is embedded by `kaggle/build_submission_notebook.py` so this notebook is self-contained when submitted.


In [ ]:
from pathlib import Path
import sys

PACKAGE_DIR = Path('/kaggle/working/arc_solver')
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
SOURCES = {'__init__.py': '"""ARC-AGI-2 task-level transformation inference."""\n\nfrom .inference import SolveResult, solve_task\n\n__all__ = ["SolveResult", "solve_task"]\n', 'grids.py': '"""Grid validation and small immutable-grid utilities."""\n\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom typing import TypeAlias\n\nGrid: TypeAlias = tuple[tuple[int, ...], ...]\nRawGrid: TypeAlias = list[list[int]]\n\n\ndef as_grid(value: object) -> Grid:\n    """Validate a JSON-compatible ARC grid and return an immutable copy."""\n    if not isinstance(value, (list, tuple)) or not value:\n        raise ValueError("A grid must be a non-empty list of rows.")\n    rows: list[tuple[int, ...]] = []\n    width: int | None = None\n    for row in value:\n        if not isinstance(row, (list, tuple)) or not row:\n            raise ValueError("Grid rows must be non-empty lists.")\n        converted: list[int] = []\n        for cell in row:\n            if isinstance(cell, bool) or not isinstance(cell, int) or not 0 <= cell <= 9:\n                raise ValueError(f"Invalid ARC cell value: {cell!r}")\n            converted.append(cell)\n        if width is None:\n            width = len(converted)\n        elif len(converted) != width:\n            raise ValueError("Grid rows must all have the same width.")\n        rows.append(tuple(converted))\n    if len(rows) > 30 or width is None or width > 30:\n        raise ValueError("ARC grids must be no larger than 30 by 30.")\n    return tuple(rows)\n\n\ndef to_json_grid(grid: Grid) -> RawGrid:\n    return [list(row) for row in grid]\n\n\ndef shape(grid: Grid) -> tuple[int, int]:\n    return len(grid), len(grid[0])\n\n\ndef most_common_color(grid: Grid) -> int:\n    counts = Counter(cell for row in grid for cell in row)\n    return min(counts, key=lambda color: (-counts[color], color))\n\n\ndef crop(grid: Grid, top: int, left: int, bottom: int, right: int) -> Grid:\n    return tuple(row[left:right] for row in grid[top:bottom])\n\n\ndef bounding_box(grid: Grid, background: int) -> tuple[int, int, int, int] | None:\n    cells = [\n        (r, c)\n        for r, row in enumerate(grid)\n        for c, color in enumerate(row)\n        if color != background\n    ]\n    if not cells:\n        return None\n    rows, cols = zip(*cells)\n    return min(rows), min(cols), max(rows) + 1, max(cols) + 1\n', 'operations.py': '"""Small, data-independent transformations used by the synthesis baseline."""\n\nfrom __future__ import annotations\n\nfrom collections import Counter, deque\nfrom dataclasses import dataclass\nfrom functools import lru_cache\nfrom typing import Callable\n\nfrom .grids import Grid, bounding_box, crop, most_common_color, shape\n\nGridTransform = Callable[[Grid], Grid | None]\n\n\n@dataclass(frozen=True)\nclass Operation:\n    name: str\n    apply: GridTransform\n    cost: int = 1\n\n\ndef _rotate(grid: Grid, turns: int) -> Grid:\n    result = grid\n    for _ in range(turns):\n        result = tuple(tuple(row) for row in zip(*result[::-1]))\n    return result\n\n\ndef _flip_horizontal(grid: Grid) -> Grid:\n    return tuple(tuple(reversed(row)) for row in grid)\n\n\ndef _flip_vertical(grid: Grid) -> Grid:\n    return tuple(reversed(grid))\n\n\ndef _translate(grid: Grid, dx: int, dy: int) -> Grid:\n    height, width = shape(grid)\n    background = most_common_color(grid)\n    rows = [[background] * width for _ in range(height)]\n    for r in range(height):\n        for c in range(width):\n            nr, nc = r + dy, c + dx\n            if 0 <= nr < height and 0 <= nc < width:\n                rows[nr][nc] = grid[r][c]\n    return tuple(tuple(row) for row in rows)\n\n\ndef _infer_translations(examples: list[tuple[Grid, Grid]]) -> list[tuple[int, int]]:\n    source, target = examples[0]\n    if shape(source) != shape(target):\n        return []\n    background = most_common_color(source)\n    anchors = [\n        (r, c, source[r][c])\n        for r, row in enumerate(source)\n        for c in range(len(row))\n        if source[r][c] != background\n    ]\n    if not anchors:\n        return []\n    color_counts = Counter(color for _, _, color in anchors)\n    anchor_color = min(color_counts, key=lambda color: (color_counts[color], color))\n    anchor_row, anchor_col, _ = next(item for item in anchors if item[2] == anchor_color)\n    target_positions = [\n        (r, c)\n        for r, row in enumerate(target)\n        for c, color in enumerate(row)\n        if color == anchor_color\n    ]\n    return sorted(\n        (col - anchor_col, row - anchor_row)\n        for row, col in target_positions\n        if (row, col) != (anchor_row, anchor_col)\n        and all(\n            _translate(example, col - anchor_col, row - anchor_row) == expected\n            for example, expected in examples\n        )\n    )\n\n\ndef _background_candidates(examples: list[tuple[Grid, Grid]]) -> list[int]:\n    candidates = {0}\n    for grid, _ in examples:\n        candidates.add(most_common_color(grid))\n        border = (\n            list(grid[0])\n            + list(grid[-1])\n            + [row[0] for row in grid[1:-1]]\n            + [row[-1] for row in grid[1:-1]]\n        )\n        if border:\n            counts = Counter(border)\n            candidates.add(min(counts, key=lambda color: (-counts[color], color)))\n    return sorted(candidates)\n\n\n@lru_cache(maxsize=256)\ndef _isolate_color(grid: Grid, color: int) -> Grid:\n    background = most_common_color(grid)\n    return tuple(\n        tuple(cell if cell == color else background for cell in row)\n        for row in grid\n    )\n\n\ndef _remove_color(grid: Grid, color: int) -> Grid:\n    background = most_common_color(grid)\n    return tuple(\n        tuple(background if cell == color else cell for cell in row)\n        for row in grid\n    )\n\n\ndef _components(\n    grid: Grid,\n    background: int,\n) -> tuple[tuple[int, int, int, int, frozenset[tuple[int, int]]], ...]:\n    height, width = shape(grid)\n    seen: set[tuple[int, int]] = set()\n    components: list[tuple[int, int, int, int, frozenset[tuple[int, int]]]] = []\n    for r in range(height):\n        for c in range(width):\n            color = grid[r][c]\n            if color == background or (r, c) in seen:\n                continue\n            queue = deque([(r, c)])\n            seen.add((r, c))\n            cells: set[tuple[int, int]] = set()\n            while queue:\n                cr, cc = queue.popleft()\n                cells.add((cr, cc))\n                for nr, nc in ((cr - 1, cc), (cr + 1, cc), (cr, cc - 1), (cr, cc + 1)):\n                    if (\n                        0 <= nr < height\n                        and 0 <= nc < width\n                        and grid[nr][nc] == color\n                        and (nr, nc) not in seen\n                    ):\n                        seen.add((nr, nc))\n                        queue.append((nr, nc))\n            rows, cols = zip(*cells)\n            components.append(\n                (color, len(cells), min(rows), min(cols), frozenset(cells))\n            )\n    return tuple(components)\n\n\ndef _component_crop(\n    grid: Grid,\n    background: int,\n    ordering: str,\n    isolate: bool,\n) -> Grid | None:\n    components = _components(grid, background)\n    if not components:\n        return None\n    sort_keys = {\n        "largest": lambda obj: (-obj[1], obj[2], obj[3], obj[0]),\n        "smallest": lambda obj: (obj[1], obj[2], obj[3], obj[0]),\n        "topmost": lambda obj: (obj[2], obj[3], -obj[1], obj[0]),\n        "bottommost": lambda obj: (-obj[2], obj[3], -obj[1], obj[0]),\n        "leftmost": lambda obj: (obj[3], obj[2], -obj[1], obj[0]),\n        "rightmost": lambda obj: (-obj[3], obj[2], -obj[1], obj[0]),\n    }\n    selected = sorted(components, key=sort_keys[ordering])[0]\n    _, _, top, left, cells = selected\n    bottom = max(r for r, _ in cells) + 1\n    right = max(c for _, c in cells) + 1\n    if isolate:\n        return tuple(\n            tuple(grid[r][c] if (r, c) in cells else background for c in range(left, right))\n            for r in range(top, bottom)\n        )\n    return crop(grid, top, left, bottom, right)\n\n\ndef _learn_color_map(examples: list[tuple[Grid, Grid]]) -> dict[int, int] | None:\n    mapping: dict[int, int] = {}\n    for source, target in examples:\n        if shape(source) != shape(target):\n            return None\n        for srow, trow in zip(source, target):\n            for old, new in zip(srow, trow):\n                if old in mapping and mapping[old] != new:\n                    return None\n                mapping[old] = new\n    return mapping if any(old != new for old, new in mapping.items()) else None\n\n\ndef build_operations(examples: list[tuple[Grid, Grid]]) -> list[Operation]:\n    operations: list[Operation] = [\n        Operation("identity", lambda grid: grid, 0),\n        Operation("rotate_90", lambda grid: _rotate(grid, 1)),\n        Operation("rotate_180", lambda grid: _rotate(grid, 2)),\n        Operation("rotate_270", lambda grid: _rotate(grid, 3)),\n        Operation("flip_horizontal", _flip_horizontal),\n        Operation("flip_vertical", _flip_vertical),\n        Operation("transpose", lambda grid: tuple(tuple(row) for row in zip(*grid))),\n        Operation("anti_transpose", lambda grid: _rotate(_flip_horizontal(grid), 1)),\n    ]\n\n    color_map = _learn_color_map(examples)\n    if color_map is not None:\n        operations.append(\n            Operation(\n                "global_color_map",\n                lambda grid, mapping=color_map: tuple(\n                    tuple(mapping.get(cell, cell) for cell in row) for row in grid\n                ),\n                2,\n            )\n        )\n\n    for background in _background_candidates(examples):\n        operations.append(\n            Operation(\n                f"crop_background_{background}",\n                lambda grid, bg=background: (\n                    None\n                    if (box := bounding_box(grid, bg)) is None\n                    else crop(grid, *box)\n                ),\n                2,\n            )\n        )\n        for ordering in ("largest", "smallest", "topmost", "bottommost", "leftmost", "rightmost"):\n            for isolate in (False, True):\n                operations.append(\n                    Operation(\n                        f"crop_{ordering}_component_bg{background}_isolate{int(isolate)}",\n                        lambda grid, bg=background, order=ordering, only=isolate: _component_crop(\n                            grid, bg, order, only\n                        ),\n                        3,\n                    )\n                )\n\n    colors = sorted({cell for source, _ in examples for row in source for cell in row})\n    for color in colors:\n        operations.append(\n            Operation(\n                f"isolate_color_{color}",\n                lambda grid, selected=color: _isolate_color(grid, selected),\n                2,\n            )\n        )\n        operations.append(\n            Operation(\n                f"remove_color_{color}",\n                lambda grid, removed=color: _remove_color(grid, removed),\n                2,\n            )\n        )\n        operations.append(\n            Operation(\n                f"crop_color_{color}",\n                lambda grid, selected=color: (\n                    None\n                    if (box := bounding_box(grid, selected)) is None\n                    else crop(grid, *box)\n                ),\n                2,\n            )\n        )\n\n    for dx, dy in _infer_translations(examples):\n        operations.append(\n            Operation(\n                f"translate_x{dx}_y{dy}",\n                lambda grid, x=dx, y=dy: _translate(grid, x, y),\n                2,\n            )\n        )\n\n    for factor_y in range(2, 4):\n        for factor_x in range(2, 4):\n            operations.extend(\n                [\n                    Operation(\n                        f"tile_{factor_y}x{factor_x}",\n                        lambda grid, fy=factor_y, fx=factor_x: tuple(\n                            row\n                            for _ in range(fy)\n                            for source_row in grid\n                            for row in (source_row * fx,)\n                        ),\n                        3,\n                    ),\n                    Operation(\n                        f"scale_{factor_y}x{factor_x}",\n                        lambda grid, fy=factor_y, fx=factor_x: tuple(\n                            expanded_row\n                            for source_row in grid\n                            for expanded_row in (\n                                tuple(cell for cell in source_row for _ in range(fx)),\n                            )\n                            for _ in range(fy)\n                        ),\n                        3,\n                    ),\n                ]\n            )\n    return operations\n', 'inference.py': '"""Task-level program synthesis and prediction ranking."""\n\nfrom __future__ import annotations\n\nfrom collections import defaultdict\nfrom dataclasses import dataclass\nfrom typing import Any\n\nfrom .grids import Grid, as_grid\nfrom .operations import Operation, build_operations\n\n\n@dataclass(frozen=True)\nclass Program:\n    operations: tuple[Operation, ...]\n\n    @property\n    def cost(self) -> int:\n        return sum(operation.cost for operation in self.operations)\n\n    @property\n    def name(self) -> str:\n        return " -> ".join(operation.name for operation in self.operations)\n\n    def apply(self, grid: Grid) -> Grid | None:\n        result: Grid | None = grid\n        for operation in self.operations:\n            if result is None:\n                return None\n            result = operation.apply(result)\n            if result is not None and (\n                not result\n                or not result[0]\n                or len(result) > 30\n                or len(result[0]) > 30\n                or any(len(row) != len(result[0]) for row in result)\n                or any(cell not in range(10) for row in result for cell in row)\n            ):\n                return None\n        return result\n\n\n@dataclass(frozen=True)\nclass SolveResult:\n    attempt_1: Grid\n    attempt_2: Grid\n    program_names: tuple[str, ...]\n    consistent_program_count: int\n\n\ndef _candidate_programs(\n    examples: list[tuple[Grid, Grid]],\n    max_depth: int,\n) -> list[Program]:\n    operations = build_operations(examples)\n    first_input, first_output = examples[0]\n    one_step: list[tuple[Operation, Grid | None]] = [\n        (operation, operation.apply(first_input)) for operation in operations\n    ]\n    exact_programs: list[Program] = []\n    for operation, result in one_step:\n        if result == first_output:\n            exact_programs.append(Program((operation,)))\n\n    if max_depth >= 2:\n        intermediates: dict[Grid, list[Operation]] = defaultdict(list)\n        for operation, result in one_step:\n            if (\n                result is not None\n                and result != first_output\n                and operation.name != "identity"\n            ):\n                intermediates[result].append(operation)\n        for intermediate, prefixes in intermediates.items():\n            for suffix in operations:\n                if suffix.name == "identity":\n                    continue\n                if suffix.apply(intermediate) == first_output:\n                    exact_programs.extend(\n                        Program((prefix, suffix)) for prefix in prefixes\n                    )\n\n    consistent: dict[tuple[str, ...], Program] = {}\n    for program in exact_programs:\n        if all(program.apply(source) == target for source, target in examples):\n            consistent[tuple(op.name for op in program.operations)] = program\n    return sorted(consistent.values(), key=lambda program: (program.cost, program.name))\n\n\ndef solve_task(task: dict[str, Any], max_depth: int = 2) -> list[SolveResult]:\n    """Infer a transformation from all demonstrations and predict each test input."""\n    if max_depth not in (1, 2):\n        raise ValueError("max_depth must be 1 or 2.")\n    raw_train = task.get("train")\n    raw_test = task.get("test")\n    if not isinstance(raw_train, list) or not raw_train:\n        raise ValueError("Task must contain at least one training pair.")\n    if not isinstance(raw_test, list) or not raw_test:\n        raise ValueError("Task must contain at least one test input.")\n\n    examples: list[tuple[Grid, Grid]] = []\n    for pair in raw_train:\n        examples.append((as_grid(pair["input"]), as_grid(pair["output"])))\n    programs = _candidate_programs(examples, max_depth)\n    test_inputs = [as_grid(pair["input"]) for pair in raw_test]\n    results: list[SolveResult] = []\n    for test_input in test_inputs:\n        votes: dict[Grid, list[Program]] = defaultdict(list)\n        for program in programs:\n            prediction = program.apply(test_input)\n            if prediction is not None:\n                votes[prediction].append(program)\n        ranked = sorted(\n            votes.items(),\n            key=lambda item: (\n                -len({program.operations[0].name for program in item[1]}),\n                min(program.cost for program in item[1]),\n                min(program.name for program in item[1]),\n            ),\n        )\n        matching_examples = {\n            target\n            for source, target in examples\n            if source == test_input\n        }\n        if len(matching_examples) == 1:\n            exact_example = next(iter(matching_examples))\n            results.append(\n                SolveResult(\n                    exact_example,\n                    exact_example,\n                    ("exact_demonstration_match",),\n                    len(programs),\n                )\n            )\n            continue\n        if ranked:\n            first = ranked[0][0]\n            second = ranked[1][0] if len(ranked) > 1 else first\n            names = tuple(\n                min((program.name for program in group), key=lambda name: (len(name), name))\n                for _, group in ranked[:2]\n            )\n            results.append(SolveResult(first, second, names, len(programs)))\n        else:\n            results.append(SolveResult(test_input, test_input, ("fallback_identity",), 0))\n    return results\n', 'submission.py': '"""Kaggle challenge-to-submission adapter and exact-match scoring helpers."""\n\nfrom __future__ import annotations\n\nfrom typing import Any\n\nfrom .grids import Grid, as_grid, to_json_grid\nfrom .inference import solve_task\n\n\ndef make_submission(challenges: dict[str, Any], max_depth: int = 2) -> dict[str, list[dict[str, Any]]]:\n    submission: dict[str, list[dict[str, Any]]] = {}\n    for task_id, task in challenges.items():\n        predictions = solve_task(task, max_depth=max_depth)\n        submission[task_id] = [\n            {\n                "attempt_1": to_json_grid(prediction.attempt_1),\n                "attempt_2": to_json_grid(prediction.attempt_2),\n            }\n            for prediction in predictions\n        ]\n    return submission\n\n\ndef exact_match_score(\n    submission: dict[str, list[dict[str, Any]]],\n    solutions: dict[str, Any],\n) -> tuple[int, int]:\n    """Return number of solved output grids and total output grids."""\n    solved = 0\n    total = 0\n    for task_id, expected_values in solutions.items():\n        predictions = submission[task_id]\n        if isinstance(expected_values, dict):\n            expected_values = expected_values.get("test", expected_values)\n        if not isinstance(expected_values, list) or len(expected_values) != len(predictions):\n            raise ValueError(f"Solution/prediction count mismatch for task {task_id}.")\n        for expected, prediction in zip(expected_values, predictions):\n            target: Grid = as_grid(expected)\n            total += 1\n            if (\n                as_grid(prediction["attempt_1"]) == target\n                or as_grid(prediction["attempt_2"]) == target\n            ):\n                solved += 1\n    return solved, total\n'}
for filename, source in SOURCES.items():
    (PACKAGE_DIR / filename).write_text(source, encoding='utf-8')
sys.path.insert(0, '/kaggle/working')


In [ ]:
import json
from pathlib import Path
from arc_solver.submission import make_submission

input_files = list(Path('/kaggle/input').rglob('arc-agi_test_challenges.json'))
if len(input_files) != 1:
    raise FileNotFoundError(
        f'Expected exactly one arc-agi_test_challenges.json; found {len(input_files)}.'
    )
with input_files[0].open(encoding='utf-8') as stream:
    challenges = json.load(stream)
submission = make_submission(challenges)
submission_path = Path('/kaggle/working/submission.json')
with submission_path.open('w', encoding='utf-8') as stream:
    json.dump(submission, stream, separators=(',', ':'))
print(f'Wrote {len(submission)} task predictions to {submission_path}')
